# 🟢 미션 1 (필수) — 내 프롬프트로 옵션 3개 비교

수업 `## 5` 에서 한 것을 **내가 고른 문장**으로 한다.

## 낼 것

1. 세 가지 결과가 나란히 나온 **표 캡처**
2. 한 줄 — **어느 것이 제일 읽을 만했나** (아래 제출 칸)

In [1]:
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

MODEL_ID = "skt/kogpt2-base-v2"

# ⚠️ AutoTokenizer 로 열면 한글이 깨진다 (수업 `## 0` 참고)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_ID,
    bos_token="</s>", eos_token="</s>",
    unk_token="<unk>", pad_token="<pad>", mask_token="<mask>",
)
model = GPT2LMHeadModel.from_pretrained(MODEL_ID).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("준비 완료 :", device)

/home/student/llm-practice/c2-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 21301.04it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


준비 완료 : cuda


## 1. 내 프롬프트를 정한다

이야기의 첫머리처럼 **뒤가 열려 있는 문장**이 재미있다.
예: `"밤늦게 도착한 기차역에는"` · `"할머니가 들려준 이야기 중에"` · `"처음 회사에 출근한 날"`

In [2]:
내_프롬프트 = "밤늦게 도착한 기차역에는"        # ← 여기를 내 문장으로 바꾼다

print("내 프롬프트 :", 내_프롬프트)
print("토큰        :", tokenizer.tokenize(내_프롬프트))

내 프롬프트 : 밤늦게 도착한 기차역에는
토큰        : ['▁밤', '늦게', '▁도착한', '▁기', '차', '역', '에는']


## 2. 세 가지 옵션으로 각각 생성한다

In [3]:
def generate(**options):
    ids = tokenizer.encode(내_프롬프트, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=60,
                             pad_token_id=tokenizer.pad_token_id, **options)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(내_프롬프트):].strip()


SETTINGS = {
    "greedy (1등만)":      dict(do_sample=False),
    "top_p=0.92":          dict(do_sample=True, top_p=0.92, top_k=0),
    "temperature=1.3":     dict(do_sample=True, temperature=1.3, top_k=50),
}

results = {}
for label, options in SETTINGS.items():
    torch.manual_seed(0)          # 같은 조건에서 비교하려고 씨앗을 고정한다
    results[label] = generate(**options)

for label, text in results.items():
    print("=" * 60)
    print(f"[{label}]")
    print(text)
    print()

[greedy (1등만)]
기차역 직원들이 나와 승객들을 맞이하고 있었다.
기차역 직원들은 기차역 직원들에게 "기차역 직원들이 기차역 직원들에게 '우리도 기차역 직원들이니까 기차역 직원들이 기차역 직원들을 위해 열심히 일해달라'고 부탁했

[top_p=0.92]
훤칠한 대형 버스가 보였다.
그 택시는 아홉 살배기 딸은 보고 싶은 산딸이 살포시는 재촉에 나서기 위해 필요한 버스를 타고 5분쯤 달려 떠났다.
신호등이 없는 길이라고 노인은 위기관리실에 연락해 가게를 알아본 뒤 곧장 백화점으로

[temperature=1.3]
10여명의 승객을 태우고 오전시간을 기다려온 택시 5대가 바짝 섰다.
하지만 오전 11시30분께는 모두 다친 택시기사가 있는가 하면 택시기사에 맞은편에 타고 있던 또 다른 승객이 흉기를 든 채 달아나면서 한때 운전자 1명이 사망하고 한 승객이 다쳤다.



## 3. 읽어 보고 판단한다

볼 것 세 가지 —

- **같은 말을 맴도는가?** (반복 붕괴)
- **말이 되는가?** 문장이 이어지는가
- **재미있는가?** 뻔한가, 엉뚱한가

> 결과가 마음에 안 들면 `내_프롬프트` 를 바꿔서 다시 돌려 본다.
> 짧은 프롬프트일수록 모델이 헤맨다. 한 문장 정도가 적당하다.

## 제출

위 세 결과 표를 캡처하고, 아래 한 줄을 채워서 함께 낸다.

```
내 프롬프트 :
제일 읽을 만했던 것 :            (greedy / top_p / temperature 중 하나)
그렇게 본 이유 한 줄 :
```

## 🔵 더 해 보고 싶다면 — 정해진 답 없음

여기부터는 제출물이 아니다. **시간이 남거나 재미있으면** 골라서 해 본다.
코드는 위 칸을 그대로 쓰고 값만 바꾸면 된다.

- `temperature` 만 바꿔 가며 열 번쯤 돌려 보고, **제일 이상한 문장**을 찾아 본다.
  어느 값부터 말이 안 되기 시작하나?
- `top_p` 를 **0.1과 0.99** 로 극단까지 줘 본다. 각각 무엇이 문제인가?
- 프롬프트를 **한 단어**로 줄여 본다 (`"밤"`). 모델이 헤매는 게 보이나?
- 같은 프롬프트를 **존댓말과 반말**로 각각 넣어 본다. 이어 쓰는 말투가 달라지나?
- 뉴스 문투로 시작해 본다 (`"어제 오후 서울 도심에서"`). 갑자기 그럴듯해지는가?

> 재미있는 결과가 나오면 **채팅에 붙여 준다.** 다 같이 본다.